**Import packages and libraries**

In [6]:
# Install libraries
!pip install gensim
!pip install scikit-learn matplotlib
!pip install transformers torch
!pip install datasets
!pip install -U sentence-transformers
!pip install -U accelerate
!pip install plotly
!pip install vaderSentiment
!pip install sentence-transformers
!pip install xgboost

In [7]:
# Import necessary packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
import pandas as pd
import numpy as np
import json
import re

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

import networkx as nx
from networkx.algorithms import community
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt

from collections import Counter
from imblearn.over_sampling import RandomOverSampler
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, cohen_kappa_score, accuracy_score
from sklearn.utils import resample
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

import xgboost as xgb

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


**Import the git files**

In [8]:
!git clone https://github.com/coralie-sorbet/project_web_mining.git

fatal: destination path 'project_web_mining' already exists and is not an empty directory.


In [9]:
path_data = '/content/project_web_mining/database/Everything/database_formated_for_NetworkX.graphml'

graph = nx.read_graphml(path_data)

**Extract types of nodes and priorities**

In [10]:
# Extract different kinds of nodes
unique_labels = set(data.get("labels") for _, data in graph.nodes(data=True))

print("Distincts types of nodes using 'labels' attribut:", unique_labels)

Distincts types of nodes using 'labels' attribut: {':Hashtag', ':Event', ':PostCategory', ':Tweet', ':User'}


In [11]:
# We want to see the different types of priority values of a tweet
priority_types = set(data.get("annotation_postPriority") for _, data in graph.nodes(data=True) if data.get("labels") == ":Tweet")

print(f"Number of unique values for 'annotation_postPriority': {len(priority_types)}")
print(f"Unique values of 'annotation_postPriority': {priority_types}")

Number of unique values for 'annotation_postPriority': 5
Unique values of 'annotation_postPriority': {'Low', 'Unknown', 'Critical', 'High', 'Medium'}


In [12]:
# We want to see the distribution of the priority values
tweet_priorities = [data['annotation_postPriority'] for _, data in graph.nodes(data=True) if data.get('labels') == ":Tweet"]

# Count occurrences of each annotation_postPriority
priority_counts = Counter(tweet_priorities)

# Display the counts
print(priority_counts)

Counter({'Low': 25407, 'Unknown': 19323, 'Medium': 6629, 'High': 4155, 'Critical': 472})


In [13]:
# We are only interested in the Low, Medium and High categories, so we can get rid of the Unknown and Critical ones
nodes_to_remove = [
    node for node, data in graph.nodes(data=True)
    if data.get('labels') == ":Tweet" and data.get('annotation_postPriority') in ["Unknown", "Critical"]
]
graph.remove_nodes_from(nodes_to_remove)

In [14]:
import random

# Get all nodes from the graph
#all_nodes = list(graph.nodes())

# Select a random sample of 10,000 nodes (or all if fewer than 10,000 exist)
#subset_size = min(10000, len(all_nodes))  # Avoid errors if graph has fewer nodes
#random_nodes = random.sample(all_nodes, subset_size)

# Create a subgraph with only the selected nodes
#graph = graph.subgraph(random_nodes).copy()


**Feature extraction and creation**

In [15]:
# Link tweets to user information
# Initialize lists to store data
tweet_ids = []
user_ids = []
user_names = []
followers_counts = []
listed = []
statuses = []
friends = []
verified = []
favourites = []
priorities = []
texts = []
favs = []
sens = []
annots = []
retweets = []
dates = []

# Iterate over edges to extract information from "POSTED" relationships
for u, v, edge_data in graph.edges(data=True):
    if edge_data.get("label") == "POSTED":
        tweet_id = graph.nodes[v].get("id_str")  # Tweet ID
        user_id = graph.nodes[u].get("id")  # User ID
        user_name = graph.nodes[u].get("name")  # User name
        user_favourites = graph.nodes[u].get("favourites_count", 0)
        user_listed = graph.nodes[u].get("listed_count", 0)
        user_friends = graph.nodes[u].get("friends_count", 0)
        user_statuses = graph.nodes[u].get("statuses_count", 0)
        user_verified = graph.nodes[u].get("isVerified")
        followers_count = graph.nodes[u].get("followers_count", 0)  # Followers count
        priority = graph.nodes[v].get("annotation_postPriority")  # Priority category
        text = graph.nodes[v].get("text", "")  # Tweet content (default empty string if missing)
        sensitive = graph.nodes[v].get("possibly_sensitive")
        retweet = graph.nodes[v].get("retweet_count")
        fav = graph.nodes[v].get("favourite_count")
        annot = graph.nodes[v].get("annotation_num_judgements")
        date = graph.nodes[v].get("created_at")

        # Append data to lists
        tweet_ids.append(tweet_id)
        user_ids.append(user_id)
        user_names.append(user_name)
        followers_counts.append(followers_count)
        listed.append(user_listed)
        statuses.append(user_statuses)
        friends.append(user_friends)
        verified.append(user_verified)
        favourites.append(user_favourites)
        priorities.append(priority)
        texts.append(text)
        annots.append(annot)
        sens.append(sensitive)
        favs.append(fav)
        retweets.append(retweet)
        dates.append(date)

# Convert lists to DataFrame
df_tweets = pd.DataFrame({
    "tweet_id": tweet_ids,
    "user_id": user_ids,
    "user_name": user_names,
    "listed_count": listed,
    "statuses_count": statuses,
    "friends_count": friends,
    "isVerified": verified,
    "favourites_count": favourites,
    "followers_count": followers_counts,
    "annotation_postPriority": priorities,
    "text": texts,  # Include tweet content
    "annotation_num_judgements": annot,
    "possibly_sensitive": sens,
    "favourite_count": favs,
    "retweet_count": retweet,
    "created_at": dates
})

# Link tweets to hashtags
tweet_ids_ht = []
hashtags_list = []
hashtags_count_list = []

for u, v, edge_data in graph.edges(data=True):
    if edge_data.get("label") == "HAS_HASHTAG":
        tweet_id = graph.nodes[u].get("id_str")  # The tweet id
        hashtag = graph.nodes[v].get("id")  # The hashtag id
        hashtag_count = graph.nodes[v].get("occurences", 0)  # Hashtag occurrence count

        tweet_ids_ht.append(tweet_id)
        hashtags_list.append(hashtag)
        hashtags_count_list.append(hashtag_count)

# Create DataFrame for hashtags
df_hashtags = pd.DataFrame({
    "tweet_id": tweet_ids_ht,
    "hashtag": hashtags_list,
    "hashtag_count": hashtags_count_list
})

# Link tweets to events
tweet_ids_event = []
event_type = []
event_ids = []

for u, v, edge_data in graph.edges(data=True):
    if edge_data.get("label") == "IS_ABOUT":
        tweet_id_event = graph.nodes[u].get("id_str")  # The tweet id
        event_id = graph.nodes[v].get("id")
        event = graph.nodes[v].get("eventType")  # The event

        tweet_ids_event.append(tweet_id_event)
        event_ids.append(event_id)
        event_type.append(event)

df_events = pd.DataFrame({
    "tweet_id": tweet_ids_event,
    "event_id": event_ids,
    "eventType": event_type
})

# Merge tweet, event, user and hashtag data using an outer join to preserve all tweets
df_final = pd.merge(df_tweets, df_hashtags, on="tweet_id", how="outer")
df_final = pd.merge(df_final, df_events, on="tweet_id", how="outer")
print(df_final.head())

              tweet_id             user_id             user_name  \
0  1100425366944935936                 NaN                   NaN   
1  1100911297133187072  949412899860213760          PoemwaLeezve   
2  1100953738242596864            16273831  The Voice of America   
3  1117608991540912129                 NaN                   NaN   
4  1117886666348224512           802064401         PHIVOLCS-DOST   

   listed_count  statuses_count  friends_count isVerified  favourites_count  \
0           NaN             NaN            NaN        NaN               NaN   
1           0.0         12023.0          195.0      False               0.0   
2        4638.0        114542.0          238.0       True              66.0   
3           NaN             NaN            NaN        NaN               NaN   
4         690.0         21699.0           21.0       True              24.0   

   followers_count annotation_postPriority  \
0              NaN                     NaN   
1            768.0      

In [16]:
# Mentions of users in a tweet
tweet_ids = []
user_mentioned = []
user_ids = []
mention_times = []

for u, v, edge_data in graph.edges(data=True):
    if edge_data.get("label") == "MENTIONS":
        mentioned_user_id = graph.nodes[v].get("id")
        user_mentioned.append(mentioned_user_id)

        tweet_id = None
        user_id = None
        mention_time = 0

        if graph.nodes[u].get("labels") == ":Tweet":
            tweet_id = graph.nodes[u].get("id")

        if graph.nodes[u].get("labels") == ":User":
            user_id = graph.nodes[u].get("id")
            mention_time = edge_data.get("times")

        # Append to lists (ensuring all lists have the same length)
        tweet_ids.append(tweet_id)
        user_ids.append(user_id)
        mention_times.append(mention_time)

# Create DataFrame
df_mentions = pd.DataFrame({
    "tweet_id": tweet_ids,
    "user_id": user_ids,
    "mentioned_user": user_mentioned,
    "mention_times": mention_times
})

print(df_mentions["mention_times"].unique())

[1 0 2 4 5]


In [17]:
# User to Events link
users = []
events = []
event_ids = []
for u, v, edge_data in graph.edges(data=True):
    user_id = graph.nodes[u].get("id")
    event_id = graph.nodes[v].get("eventType")

    if edge_data.get("label") == "TALKS_ABOUT":
      user_id = graph.nodes[u].get("id")
      event_id = graph.nodes[v].get("id")
      event_type = graph.nodes[v].get("eventType")

    users.append(user_id)
    events.append(event_type)
    event_ids.append(event_id)

df_user_event = pd.DataFrame({
    "user_id": users,
    "event_talked_about_by_user": event_ids,
    "eventType_talked_about_by_user": events
})


[None 'philippinesEarthquake2019' 'nepalEarthquake2015'
 'cycloneKenneth2019' 'earthquake' 'flood' 'fireColorado2012'
 'albertaFloods2013' 'typhoon' 'wildfire' 'manilaFloods2013']


In [18]:
df_mentions_tweet = df_mentions[["tweet_id", "mentioned_user"]]
df_mentions_tweet.loc[:, "tweet_id"] = pd.to_numeric(df_mentions_tweet["tweet_id"], errors="coerce").astype("Int64")
df_final.loc[:, "tweet_id"] = pd.to_numeric(df_final["tweet_id"], errors="coerce").astype("Int64")
df_mentions_tweet = df_mentions_tweet[~df_mentions_tweet["tweet_id"].isna()]
df_final = pd.merge(df_final, df_mentions_tweet, on="tweet_id", how="outer")

In [19]:
df_mentions_user = df_mentions[["user_id", "mentioned_user", "mention_times"]]
df_mentions_user.loc[:, "user_id"] = pd.to_numeric(df_mentions_user["user_id"], errors="coerce").astype("Int64")
df_mentions_user.loc[:, "mentioned_user"] = pd.to_numeric(df_mentions_user["mentioned_user"], errors="coerce").astype("Int64")
df_mentions_user.loc[:, "mention_times"] = pd.to_numeric(df_mentions_user["mention_times"], errors="coerce").astype("Int64")
df_final.loc[:, "user_id"] = pd.to_numeric(df_final["user_id"], errors="coerce").astype("Int64")

# How many users someone mentions to identify users who interact with many different people
df_user_mentions_count = df_mentions_user.groupby("user_id")["mentioned_user"].nunique().reset_index()
df_user_mentions_count.rename(columns={"mentioned_user": "number_users_mentioned"}, inplace=True)

# Total mentions recieved: How often a user is mentioned
df_mentions_received = df_mentions_user.groupby("mentioned_user")["mention_times"].sum().reset_index()
df_mentions_received.rename(columns={"mention_times": "total_mentions_received"}, inplace=True)
df_mentions_received.rename(columns={"mentioned_user": "user_id"}, inplace=True)

# Mentions given vs recieved ratio
# > 1 : user mentions a lot of people but isn't mentioned much (marketer)
# < 1 : user is mentioned often but doesn't engage much (influencer)
df_mentions_given = df_mentions_user.groupby("user_id")["mention_times"].sum().reset_index()
df_mentions_given.rename(columns={"mention_times": "total_mentions_given"}, inplace=True)

df_mention_ratio = df_mentions_given.merge(df_mentions_received, on="user_id", how="outer").fillna(0)
df_mention_ratio["mention_ratio"] = df_mention_ratio["total_mentions_given"] / (df_mention_ratio["total_mentions_received"] + 1)
df_mention_ratio = df_mention_ratio[["user_id", "mention_ratio"]]

# Mutual mentions : high score = strong connections
df_reciprocal_mentions = df_mentions_user.merge(df_mentions_user, left_on=["user_id", "mentioned_user"], right_on=["mentioned_user", "user_id"])
df_reciprocal_mentions["reciprocity_score"] = df_reciprocal_mentions["mention_times_x"] + df_reciprocal_mentions["mention_times_y"]
df_reciprocal_mentions.rename(columns={"user_id_x": "user_id"}, inplace=True)
df_reciprocal_mentions = df_reciprocal_mentions[["user_id", "reciprocity_score"]]

df_user_mentions_count = df_user_mentions_count.merge(df_mentions_received, on="user_id", how="outer").fillna(0)
df_user_mentions_count = df_user_mentions_count.merge(df_mentions_given, on="user_id", how="outer").fillna(0)
df_user_mentions_count = df_user_mentions_count.merge(df_mention_ratio, on="user_id", how="outer").fillna(0)
df_user_mentions_count = df_user_mentions_count.merge(df_reciprocal_mentions, on="user_id", how="outer").fillna(0)

df_final = df_final.merge(df_user_mentions_count, on="user_id", how="outer")

                 user_id  number_users_mentioned  total_mentions_received  \
0                 428333                     0.0                      9.0   
1                5618162                     1.0                      0.0   
2                6519522                     0.0                      2.0   
3                7424642                     0.0                      1.0   
4                8923182                     0.0                      1.0   
..                   ...                     ...                      ...   
254   938796641901039617                     0.0                      1.0   
255   962734359689117696                     0.0                      0.0   
256   985522863846207488                     1.0                      0.0   
257   991384624017354752                     0.0                      1.0   
258  1000935447042834432                     1.0                      0.0   

     total_mentions_given  mention_ratio  reciprocity_score  
0            

<ipython-input-19-37265162808f>:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_user_mentions_count = df_user_mentions_count.merge(df_reciprocal_mentions, on="user_id", how="outer").fillna(0)


In [21]:
df_final["hashtag"]= df_final["hashtag"].astype(str)
df_final["event_id"]= df_final["event_id"].astype(str)
df_final["eventType"]= df_final["eventType"].astype(str)
df_final["mentioned_user"]= df_final["mentioned_user"].astype(str)
df_final["annotation_postPriority"]= df_final["annotation_postPriority"].astype(str)
df_final["isVerified"]= df_final["isVerified"].astype(bool)
df_final["text"]= df_final["text"].astype(str)
df_final["possibly_sensitive"]= df_final["possibly_sensitive"].astype(bool)
df_final["favourite_count"]= df_final["favourite_count"].astype(float)
df_final["created_at"]= df_final["created_at"].astype(str)

In [22]:
columns = df_final.columns
for col in columns:
  if df_final[col].dtype == "str":
    df_final[col] = df_final[col].fillna("Unknown")
  if df_final[col].dtype == "float64":
    df_final[col] = df_final[col].fillna(0)


In [23]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Select features (excluding 'followers_count')
X = df_final[["friends_count", "favourites_count", "statuses_count", "listed_count", "isVerified", 'mention_ratio', 'reciprocity_score']]

# Standardize data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
pca = PCA(n_components=1)  # Keep only the first principal component
df_final["popularity_score"] = pca.fit_transform(X_scaled)

# Normalize to [0,1] for better interpretability
df_final["popularity_score"] = (df_final["popularity_score"] - df_final["popularity_score"].min()) / \
                         (df_final["popularity_score"].max() - df_final["popularity_score"].min())

#print(df_final)


In [24]:
# Create the column has_emoji

# Emoji pattern (covers most emojis)
emoji_pattern = re.compile("["
    "\U0001F600-\U0001F64F"  # Emoticons
    "\U0001F300-\U0001F5FF"  # Symbols & pictographs
    "\U0001F680-\U0001F6FF"  # Transport & map symbols
    "\U0001F700-\U0001F77F"  # Alchemical symbols
    "\U0001F780-\U0001F7FF"  # Geometric shapes
    "\U0001F800-\U0001F8FF"  # Supplemental arrows
    "\U0001F900-\U0001F9FF"  # Supplemental symbols and pictographs
    "\U0001FA00-\U0001FA6F"  # Chess symbols, etc.
    "\U0001FA70-\U0001FAFF"  # More symbols
    "\U00002702-\U000027B0"  # Dingbats
    "\U000024C2-\U0001F251"
"]+", flags=re.UNICODE)

# Function to extract emojis from text
def extract_emojis(text):
    return emoji_pattern.findall(text) if isinstance(text, str) else []

# Apply functions to create new columns
df_final['has_emoji'] = df_final['text'].apply(lambda x: bool(emoji_pattern.search(str(x))))
df_final['emoji_count'] = df_final['text'].apply(lambda x: len(extract_emojis(str(x))))

In [25]:
# We want to detect opinianated words. We will use VADER Scores
# VADER gives four sentiment scores:
# pos (0 to 1), neg (0 to 1), neu (0 to 1), compound (-1 to 1)
# If compound > 0.05 :positive, if compound < -0.05 negative, if -0.05 < compound < 0.05 neutral

# Create a column that says True if opinianated words are used in the tweet

# Initialize the Sentiment Analyzer
sia = SentimentIntensityAnalyzer()

# Function to classify sentiment
def classify_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

df_final['opinion'] = df_final['text'].apply(classify_sentiment)

In [26]:
import nltk
import gensim.downloader as api
from nltk.tokenize import word_tokenize

def get_word_embeddings():
  # Download and load a pre-trained Word2Vec model
  nltk.download('punkt_tab')
  model = api.load("word2vec-google-news-300")
  return model

def get_tweet_word_embedding(doc, model):
    def preprocess_text(text):
      return word_tokenize(text.lower())

    words = preprocess_text(doc)
    word_vectors = []
    for word in words:
        if word in model:
            word_vectors.append(model[word])

    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)

    document_embedding = np.mean(word_vectors, axis=0)
    return document_embedding
w2v_model = get_word_embeddings()

tweet_embeddings = []  # Store embeddings
texts = df_final["text"]
for tweet_text in texts:
    tweet_vector = get_tweet_word_embedding(tweet_text, w2v_model)
    tweet_embeddings.append(tweet_vector)

df_final["tweet_embedding"] = tweet_embeddings

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [32]:
df_final["user_name"] = df_final["user_name"].astype(str)

In [33]:
user_embeddings = []
usernames = df_final["user_name"]
# Iterate over the nodes of type 'Tweet'
for username in usernames:
    user_vector = get_tweet_word_embedding(username, w2v_model)
    user_embeddings.append(user_vector)

df_final["user_embeddings"] = user_embeddings

In [47]:
df_final = df_final.dropna(subset=["tweet_id", "user_id"])

In [35]:
df_final = df_final[df_final["annotation_postPriority"] != "Unknown"]


**Preparing the model**

In [36]:
X = df_final.drop('annotation_postPriority', axis=1)  # Features
y = df_final['annotation_postPriority']  # Target: the priority classes

# Initialize Stratified Shuffle Split (to maintain the class distribution in both train and test)
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)

# Split data
for train_index, test_index in splitter.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Combine features and target into a train/test dataset
train_data = X_train.copy()
train_data['annotation_postPriority'] = y_train

test_data = X_test.copy()
test_data['annotation_postPriority'] = y_test


In [37]:
# Check class distribution in the training and test sets
print("Training Set Class Distribution:")
print(train_data['annotation_postPriority'].value_counts())

print("\nTest Set Class Distribution:")
print(test_data['annotation_postPriority'].value_counts())


Training Set Class Distribution:
annotation_postPriority
nan       1059
Low        635
Medium     200
High        38
Name: count, dtype: int64

Test Set Class Distribution:
annotation_postPriority
nan       454
Low       273
Medium     86
High       16
Name: count, dtype: int64


In [38]:
# Separate the majority and minority classes in the training data
majority_class = train_data[train_data['annotation_postPriority'] == 'Low']  # Replace with your exact class names
minority_class = train_data[train_data['annotation_postPriority'] != 'Low']

# Oversample the minority class
minority_upsampled = resample(minority_class,
                               replace=True,
                               n_samples=len(majority_class),  # Match the size of the majority class
                               random_state=42)

# Combine majority class with oversampled minority class
train_balanced = pd.concat([majority_class, minority_upsampled])

# Shuffle the training data to ensure randomness
train_balanced = train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)


In [39]:
# Check class distribution in the training and test sets
print("Training Set Class Distribution:")
print(train_balanced['annotation_postPriority'].value_counts())

print("\nTest Set Class Distribution:")
print(test_data['annotation_postPriority'].value_counts())


Training Set Class Distribution:
annotation_postPriority
Low       635
nan       512
Medium    101
High       22
Name: count, dtype: int64

Test Set Class Distribution:
annotation_postPriority
nan       454
Low       273
Medium     86
High       16
Name: count, dtype: int64


In [40]:
# Drop non-numeric columns
X_train_numeric = X_train.select_dtypes(exclude=['object'])
X_test_numeric = X_test.select_dtypes(exclude=['object'])

# Scale the numeric features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_numeric)  # Fit and transform on train data
X_test_scaled = scaler.transform(X_test_numeric)  # Transform on test data


# Apply TF-IDF on the 'text' column
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_text = tfidf_vectorizer.fit_transform(X_train['text']).toarray()
X_test_text = tfidf_vectorizer.transform(X_test['text']).toarray()

# Now 'text' is numerically represented, we can concatenate it with the rest
X_train_final = np.concatenate([X_train_scaled, X_train_text], axis=1)
X_test_final = np.concatenate([X_test_scaled, X_test_text], axis=1)


In [ ]:
# try with smaller subset
print(len(X_train_final))
X_train_final = X_train_final[:1000]
y_train = y_train[:1000]

10000


**Prediction Models**

In [41]:
# RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)

model.fit(X_train_final, y_train)

y_pred = model.predict(X_test_final)

print("Accuracy on Test Data:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy on Test Data: 0.9541616405307599

Classification Report:
               precision    recall  f1-score   support

        High       0.00      0.00      0.00        16
         Low       0.88      1.00      0.93       273
      Medium       1.00      0.74      0.85        86
         nan       1.00      1.00      1.00       454

    accuracy                           0.95       829
   macro avg       0.72      0.69      0.70       829
weighted avg       0.94      0.95      0.94       829



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [42]:
# Logistic Regression
log_reg = LogisticRegression(max_iter=1000, random_state=42)

log_reg.fit(X_train_final, y_train)

y_pred_log = log_reg.predict(X_test_final)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print("\nClassification Report:\n", classification_report(y_test, y_pred_log))


Logistic Regression Accuracy: 0.9481302774427021

Classification Report:
               precision    recall  f1-score   support

        High       0.00      0.00      0.00        16
         Low       0.88      0.98      0.93       273
      Medium       0.94      0.76      0.84        86
         nan       1.00      1.00      1.00       454

    accuracy                           0.95       829
   macro avg       0.71      0.68      0.69       829
weighted avg       0.93      0.95      0.94       829



In [43]:
# SVM with RBF Kernel
svm_model = SVC(kernel='rbf', C=1.0, random_state=42)

svm_model.fit(X_train_final, y_train)

y_pred_svm = svm_model.predict(X_test_final)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nClassification Report:\n", classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.9529553679131484

Classification Report:
               precision    recall  f1-score   support

        High       0.00      0.00      0.00        16
         Low       0.89      0.99      0.93       273
      Medium       0.94      0.78      0.85        86
         nan       1.00      1.00      1.00       454

    accuracy                           0.95       829
   macro avg       0.71      0.69      0.70       829
weighted avg       0.94      0.95      0.94       829



In [44]:
from sklearn.preprocessing import LabelEncoder

# Initialize label encoder
label_encoder = LabelEncoder()
# XGBoost Classifier
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)  # Use transform only (not fit) for test set

xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, use_label_encoder=False, eval_metric="logloss")

xgb_model.fit(X_train_final, y_train_encoded)

y_pred_xgb = xgb_model.predict(X_test_final)

print("XGBoost Accuracy:", accuracy_score(y_test_encoded, y_pred_xgb))
print("\nClassification Report:\n", classification_report(y_test_encoded, y_pred_xgb))


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [21:13:11] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost Accuracy: 0.9565741857659831

Classification Report:
               precision    recall  f1-score   support

           0       0.25      0.06      0.10        16
           1       0.89      0.99      0.94       273
           2       0.99      0.79      0.88        86
           3       1.00      1.00      1.00       454

    accuracy                           0.96       829
   macro avg       0.78      0.71      0.73       829
weighted avg       0.95      0.96      0.95       829

